<a href="https://colab.research.google.com/github/MateusDeLimaCosta/Atividade---PySpark/blob/main/lista_exercicios_pyspark_resolvida.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Processamento de Dados Massivos — Lista de Exercícios 1
## Introdução ao PySpark

**Professor:** Alexandre Roriz

**Aluno:** Mateus de Lima Costa



In [1]:
!wget -q https://huggingface.co/datasets/alexvaroz/nyc_tripdata_2024_sample_4M/resolve/main/nyc_tripdata_2024_sample_4M.csv

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ExerciciosPySpark")
    .master("local[*]")
    .getOrCreate()
)

df = spark.read.csv(
    "nyc_tripdata_2024_sample_4M.csv",
    header=True,
    inferSchema=True
)

print("Base carregada com sucesso.")

Base carregada com sucesso.


## Questão 1

Exibir o schema inferido, as 10 primeiras linhas e o número total de linhas.

In [2]:
df.printSchema()
df.show(10, truncate=False)

total_linhas = df.count()
print(f"Número total de linhas: {total_linhas:,}")

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+-----------

## Questão 2

Selecionar as colunas pedidas e exibir as 5 primeiras linhas.

In [3]:
df_q2 = df.select(
    "VendorID",
    "tpep_pickup_datetime",
    "trip_distance",
    "fare_amount",
    "payment_type"
)
df_q2.show(5, truncate=False)

+--------+--------------------+-------------+-----------+------------+
|VendorID|tpep_pickup_datetime|trip_distance|fare_amount|payment_type|
+--------+--------------------+-------------+-----------+------------+
|1       |2024-10-01 00:59:55 |0.5          |5.1        |1           |
|1       |2024-10-01 00:08:59 |20.6         |76.5       |2           |
|2       |2024-10-01 00:18:38 |7.42         |33.1       |4           |
|2       |2024-10-01 00:20:06 |19.96        |70.0       |1           |
|1       |2024-10-01 00:09:02 |2.6          |15.6       |1           |
+--------+--------------------+-------------+-----------+------------+
only showing top 5 rows


## Questão 3

Filtrar `trip_distance > 5` e `passenger_count >= 3` e contar as corridas restantes.

In [4]:
from pyspark.sql.functions import col

df_q3 = df.filter(
    (col("trip_distance") > 5) &
    (col("passenger_count") >= 3)
)

quantidade_q3 = df_q3.count()
print(f"Quantidade de corridas após o filtro: {quantidade_q3:,}")

Quantidade de corridas após o filtro: 50,665


## Questão 4 — Resposta descritiva

Quando usamos `inferSchema=True`, o Spark analisa os valores do arquivo CSV para tentar identificar automaticamente o tipo mais adequado de cada coluna, em vez de manter todas as colunas como texto. Depois da leitura, é importante usar `printSchema()` para conferir se os tipos inferidos correspondem ao esperado.

A vantagem do `inferSchema=True` é a praticidade, porque não precisamos escrever toda a estrutura do arquivo antes da leitura. Isso é útil principalmente em exploração inicial. A desvantagem é o trabalho adicional para inferir os tipos e o risco de a inferência não representar corretamente uma coluna quando existem valores inconsistentes.

Na definição manual, usamos `StructType` e `StructField` para informar antecipadamente os nomes e tipos das colunas. Isso oferece maior controle, mantém a estrutura consistente entre execuções e evita o custo de inferência. Em arquivos com milhões de linhas e em pipelines recorrentes, essa previsibilidade é importante.

Por outro lado, o schema manual exige conhecer previamente a estrutura da base. Se um tipo for definido incorretamente, podem ocorrer conversões inválidas, valores nulos ou erros, dependendo dos dados e da configuração de leitura.

Assim, `inferSchema=True` é conveniente para exploração e protótipos, enquanto o schema manual é mais adequado quando a estrutura é conhecida e se deseja maior controle e eficiência.

## Questão 5

Agrupar por `payment_type`, calcular quantidade de corridas e receita total, ordenando pela maior receita.

In [5]:
from pyspark.sql.functions import count, sum as spark_sum, desc

resultado_q5 = (
    df.groupBy("payment_type")
      .agg(
          count("*").alias("quantidade_corridas"),
          spark_sum("total_amount").alias("receita_total")
      )
      .orderBy(desc("receita_total"))
)

resultado_q5.show(truncate=False)

+------------+-------------------+--------------------+
|payment_type|quantidade_corridas|receita_total       |
+------------+-------------------+--------------------+
|1           |3045849            |9.116799616010016E7 |
|2           |553536             |1.2987084559999354E7|
|0           |410746             |1.0123049400000528E7|
|3           |29100              |220775.24999999974  |
|4           |79511              |133192.01999999984  |
|5           |1                  |62.0                |
+------------+-------------------+--------------------+



## Questão 6

Criar `hora_embarque`, agrupar por hora e calcular tarifa média e distância média.

In [6]:
from pyspark.sql.functions import hour, avg

df_q6 = df.withColumn(
    "hora_embarque",
    hour(col("tpep_pickup_datetime"))
)

resultado_q6 = (
    df_q6.groupBy("hora_embarque")
         .agg(
             avg("fare_amount").alias("tarifa_media"),
             avg("trip_distance").alias("distancia_media")
         )
         .orderBy("hora_embarque")
)

resultado_q6.show(24, truncate=False)

+-------------+------------------+------------------+
|hora_embarque|tarifa_media      |distancia_media   |
+-------------+------------------+------------------+
|0            |19.72867660335703 |5.130178643081671 |
|1            |17.54847894641617 |3.739999483030465 |
|2            |16.426538461538446|4.542445678033303 |
|3            |17.24064640950263 |3.401675265462839 |
|4            |22.33585451861572 |11.412191651631977|
|5            |26.226564065583663|23.33988120540463 |
|6            |21.931859821807333|14.540589393296582|
|7            |19.33026953083623 |11.087329050022918|
|8            |18.511148615351107|8.533842520592145 |
|9            |18.400291333656767|5.604490502277248 |
|10           |18.56454835768904 |4.5114450807098905|
|11           |18.851552824117647|4.075729557436569 |
|12           |19.207714073999153|4.468694683646846 |
|13           |19.95819012628161 |5.262006204074442 |
|14           |20.604332332373676|4.622973056355365 |
|15           |20.7571078345

## Questão 7 — Resposta descritiva

No Spark, **transformações** criam um novo DataFrame a partir de outro, mas não executam imediatamente todo o processamento. Elas constroem o plano de execução. Já as **ações** pedem um resultado concreto e, por isso, disparam a execução do plano.

Na Questão 2, `select()` é uma transformação e `show()` é a ação. Na Questão 3, `filter()` é uma transformação e `count()` é a ação. Na Questão 5, `groupBy()`, `agg()` e `orderBy()` constroem o DataFrame agregado, enquanto `show()` dispara a execução.

Isso corresponde à **lazy evaluation (avaliação preguiçosa)**. O Spark acumula as transformações antes de executá-las. Quando uma ação é chamada, ele analisa o conjunto completo de operações, monta e otimiza o plano de execução e só então processa os dados.

A vantagem prática é permitir que o Spark evite trabalho desnecessário, combine operações e escolha uma execução mais eficiente, em vez de executar cada transformação isoladamente no momento em que ela aparece no código.

## Questão 8

Calcular `percentual_gorjeta` apenas para `total_amount > 0` e mostrar as 10 maiores.

In [7]:
df_q8 = (
    df.filter(col("total_amount") > 0)
      .withColumn(
          "percentual_gorjeta",
          (col("tip_amount") / col("total_amount")) * 100
      )
)

resultado_q8 = (
    df_q8.select(
        "VendorID",
        "total_amount",
        "tip_amount",
        "percentual_gorjeta"
    )
    .orderBy(desc("percentual_gorjeta"))
)

resultado_q8.show(10, truncate=False)

+--------+------------+----------+------------------+
|VendorID|total_amount|tip_amount|percentual_gorjeta|
+--------+------------+----------+------------------+
|2       |1.63        |5.27      |323.3128834355828 |
|2       |2.07        |3.68      |177.7777777777778 |
|2       |1.6         |2.82      |176.24999999999997|
|2       |2.33        |3.72      |159.65665236051504|
|2       |2.54        |3.76      |148.03149606299212|
|2       |3.76        |3.96      |105.31914893617022|
|2       |39.7        |40.0      |100.75566750629723|
|2       |0.08        |0.08      |100.0             |
|1       |197.0       |196.0     |99.49238578680203 |
|1       |150.0       |149.0     |99.33333333333333 |
+--------+------------+----------+------------------+
only showing top 10 rows


## Questão 9

Baixar a tabela de zonas, fazer o join por `PULocationID = LocationID`, agrupar por `Borough` e ordenar do maior para o menor.

In [8]:
!wget -q https://huggingface.co/datasets/alexvaroz/nyc_tripdata_2024_sample_4M/resolve/main/taxi_zone_lookup.csv

zonas = spark.read.csv(
    "taxi_zone_lookup.csv",
    header=True,
    inferSchema=True
)

zonas.show(5, truncate=False)

+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 5 rows


In [9]:
df_com_zonas = df.join(
    zonas,
    df.PULocationID == zonas.LocationID,
    "left"
)

resultado_q9 = (
    df_com_zonas.groupBy("Borough")
                .count()
                .orderBy(desc("count"))
)

resultado_q9.show(truncate=False)

+-------------+-------+
|Borough      |count  |
+-------------+-------+
|Manhattan    |3641752|
|Queens       |388736 |
|Brooklyn     |60200  |
|Bronx        |12702  |
|Unknown      |12172  |
|N/A          |2421   |
|EWR          |565    |
|Staten Island|195    |
+-------------+-------+



## Questão 10

O bloco abaixo mede os dois tempos. Como `groupBy()` é uma transformação e o Spark usa lazy evaluation, é necessária uma ação para medir a execução real da agregação.

In [10]:
import time

inicio = time.perf_counter()
_ = df.count()
tempo_count = time.perf_counter() - inicio

inicio = time.perf_counter()
_ = (
    df.groupBy("payment_type")
      .agg(
          count("*").alias("quantidade_corridas"),
          spark_sum("total_amount").alias("receita_total")
      )
      .collect()
)
tempo_groupby = time.perf_counter() - inicio

print(f"Tempo do count(): {tempo_count:.3f} segundos")
print(f"Tempo do groupBy + agg: {tempo_groupby:.3f} segundos")

Tempo do count(): 2.755 segundos
Tempo do groupBy + agg: 10.721 segundos


### Questão 10 — Resposta descritiva

Os tempos exatos variam conforme os recursos do Colab, a inicialização da JVM e o estado de cache. Conceitualmente, porém, o agrupamento tende a ser mais custoso.

O `count()` precisa percorrer os registros para contabilizá-los, mas não precisa reorganizar as linhas por uma chave. Já o `groupBy("payment_type")` precisa reunir registros com a mesma chave antes da agregação. Em processamento distribuído, isso pode gerar **shuffle**, isto é, redistribuição de dados entre partições e Executors.

O shuffle é caro porque pode envolver comunicação de rede, uso de disco e sincronização. Seleções e filtros normalmente podem ser executados independentemente em cada partição, sem redistribuir registros entre nós.

Por isso, mesmo trabalhando sobre o mesmo volume de entrada, operações de `groupBy()` e agregação tendem a custar mais do que `select()`, `filter()` ou uma contagem simples.